# Caso de uso: Uber

**Objetivo:** Predecir si en un requerimiento habrá algún chofer disponible


In [1]:
# Librerías

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
pd.set_option('display.max_columns', None)
warnings.filterwarnings("ignore")

In [2]:
# Cargar datos

data = pd.read_csv("../data/ncr_ride_bookings.csv")
data.head(2)

,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,Cancelled Rides by Customer,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,NaN,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI


In [3]:
# TODO: Poner en módulo utils
def limpiar_columnas(df):
    df.columns = [i.replace(" ","_").lower() for i in df.columns]
    return df

data = limpiar_columnas(data)
data["date"] = pd.to_datetime(data["date"], errors="raise").dt.date
data["time"] = pd.to_datetime(data["time"], errors="raise").dt.time

In [4]:
# TODO: Poner en un archivo config
import pandera as pa
from typing import Optional
import datetime as dt

# Creamos un esquema utilizando la API de Schema en lugar de SchemaModel

def create_uber_ride_schema():
    """
    Crea un esquema para validar el DataFrame de viajes de Uber.
    """
    # Definir las columnas obligatorias (no nulas)
    schema = pa.DataFrameSchema({
        "date": pa.Column(
            dtype=object,  
            nullable=False,
            coerce=True,
            checks=[pa.Check(lambda x: isinstance(x, dt.date), element_wise=True)]
        ),
        "time": pa.Column(
            dtype=object, 
            nullable=False,
            coerce=True,
            checks=[pa.Check(lambda x: isinstance(x, dt.time), element_wise=True)]
        ),
        "booking_id": pa.Column(
            dtype=str,
            nullable=False,
            coerce=True
        ),
        "booking_status": pa.Column(
            dtype=str,
            nullable=False,
            coerce=True
        ),
        "customer_id": pa.Column(
            dtype=str,
            nullable=False,
            coerce=True
        ),
        "vehicle_type": pa.Column(
            dtype=str,
            nullable=False,
            coerce=True
        ),
    
        "pickup_location": pa.Column(
            dtype=str,
            nullable=True,
            coerce=True
        ),
        "drop_location": pa.Column(
            dtype=str,
            nullable=True,
            coerce=True
        ),
        "avg_vtat": pa.Column(
            dtype=float,
            nullable=True,
            coerce=True
        ),
        "avg_ctat": pa.Column(
            dtype=float,
            nullable=True,
            coerce=True
        ),
        "cancelled_rides_by_customer": pa.Column(
            dtype=float,  
            nullable=True,
            coerce=True
        ),
        "reason_for_cancelling_by_customer": pa.Column(
            dtype=str,
            nullable=True,
            coerce=True
        ),
        "cancelled_rides_by_driver": pa.Column(
            dtype=float,
            nullable=True,
            coerce=True
        ),
        "driver_cancellation_reason": pa.Column(
            dtype=str,
            nullable=True,
            coerce=True
        ),
        "incomplete_rides": pa.Column(
            dtype=float,
            nullable=True,
            coerce=True
        ),
        "incomplete_rides_reason": pa.Column(
            dtype=str,
            nullable=True,
            coerce=True
        ),
        "booking_value": pa.Column(
            dtype=float,
            nullable=True,
            coerce=True
        ),
        "ride_distance": pa.Column(
            dtype=float,
            nullable=True,
            coerce=True
        ),
        "driver_ratings": pa.Column(
            dtype=float,
            nullable=True,
            coerce=True
        ),
        "customer_rating": pa.Column(
            dtype=float,
            nullable=True,
            coerce=True
        ),
        "payment_method": pa.Column(
            dtype=str,
            nullable=True,
            coerce=True
        ),
 
    }, strict=True)  # No permite columnas extra
    
    return schema

# Crear instancia del esquema
UberRideSchema = create_uber_ride_schema()

In [5]:
data = UberRideSchema.validate(data)

In [6]:
# Crear target
# Aquellos viajes donde no haya un chofer asignado
# Considerar solamente aquellos viajes en los que no hay chofer
# booking_status = "No driver found"
# TODO: Agregar esta función en el módulo utils

def crear_target(df: pd.DataFrame) -> pd.DataFrame:
    df["target"] = np.where(df["booking_status"]=="No Driver Found",1,0)
    return df

data = crear_target(data)

In [7]:
# Algunas métricas
print(data.shape)

# Rango de fechas del dataset

print(data["date"].min())
print(data["date"].max())

(150000, 22)
2024-01-01
2024-12-30


In [8]:
data.head(1)

,date,time,booking_id,booking_status,customer_id,vehicle_type,pickup_location,drop_location,avg_vtat,avg_ctat,cancelled_rides_by_customer,reason_for_cancelling_by_customer,cancelled_rides_by_driver,driver_cancellation_reason,incomplete_rides,incomplete_rides_reason,booking_value,ride_distance,driver_ratings,customer_rating,payment_method,target
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


In [9]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

# TODO: Poner en modulo feature engineering
def crear_embeddings(
        df: pd.DataFrame,
        col_name: str,
        n_embeddings: int
) -> pd.DataFrame:
    s = df[col_name].astype("string")

    cats = pd.Categorical(s)                 # categorías a partir de los datos
    codes = cats.codes + 1                   # -1 (NaN) -> 0 después de +1
    codes = np.where(cats.codes < 0, 0, codes)  # por claridad
    idx = torch.tensor(codes, dtype=torch.long)

    num_cats = len(cats.categories)          # K
    emb = nn.Embedding(num_embeddings=num_cats + 1, embedding_dim=n_embeddings, padding_idx=0)

    E = emb(idx) 
    n_embeddings_array = E.detach().cpu().numpy()
    df_embeddings = pd.DataFrame(
        n_embeddings_array,
        index = df.index
    )
    df_embeddings.columns = [f"{col_name}_{n}" for n in range(n_embeddings)]
    return pd.concat([df, df_embeddings], axis = 1)

In [10]:
data = crear_embeddings(data, "pickup_location", 32)
data = crear_embeddings(data, "drop_location", 32)

In [11]:
# Train test split

vars = [
    f"pickup_location_{i}" for i in range(32)
] + [
    f"drop_location_{i}" for i in range(32)
]

target = ["target"]

x = data[vars]
y = data[target]

In [12]:
# train test split
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state=123)


In [13]:
from catboost import CatBoostClassifier

modelo = CatBoostClassifier(random_state=123)

modelo.fit(x_train, y_train)

Learning rate set to 0.079569
0:	learn: 0.6095051	total: 63.9ms	remaining: 1m 3s
1:	learn: 0.5420017	total: 70.3ms	remaining: 35.1s
2:	learn: 0.4875178	total: 76.3ms	remaining: 25.4s
3:	learn: 0.4436233	total: 82ms	remaining: 20.4s
4:	learn: 0.4082481	total: 87.3ms	remaining: 17.4s
5:	learn: 0.3796344	total: 93.1ms	remaining: 15.4s
6:	learn: 0.3565231	total: 98.7ms	remaining: 14s
7:	learn: 0.3378079	total: 104ms	remaining: 12.9s
8:	learn: 0.3226109	total: 110ms	remaining: 12.1s
9:	learn: 0.3102614	total: 115ms	remaining: 11.4s
10:	learn: 0.3001640	total: 121ms	remaining: 10.8s
11:	learn: 0.2919004	total: 126ms	remaining: 10.4s
12:	learn: 0.2851347	total: 132ms	remaining: 10s
13:	learn: 0.2795767	total: 137ms	remaining: 9.67s
14:	learn: 0.2749898	total: 143ms	remaining: 9.38s
15:	learn: 0.2711763	total: 149ms	remaining: 9.14s
16:	learn: 0.2680333	total: 154ms	remaining: 8.93s
17:	learn: 0.2654237	total: 160ms	remaining: 8.73s
18:	learn: 0.2632808	total: 165ms	remaining: 8.5s
19:	learn: 

In [14]:
from sklearn.metrics import accuracy_score

y_pred = modelo.predict(x_test)
accuracy_score(y_test, y_pred)

0.9289666666666667

In [15]:
y_test.mean()

target    0.0709
dtype: float64